# Demo — Onde o vetor mora: Azure AI Search

**Objetivo:** guardar os vetores do acervo num índice dedicado do Azure AI Search e buscar por similaridade, no mesmo roteiro das outras cinco bases desta série, para compará-las lado a lado. Ilustra os slides 20 a 23: a pergunta que decide onde o vetor mora (slide 21), a linha do AI Search na tabela das opções (slide 22) e o algoritmo do índice (slide 23).

1. **Preparar os vetores** → os mesmos trechos e embeddings em todas as bases da série;
2. **Criar a estrutura vetorial** → um índice com campo vetorial, HNSW e KNN exaustivo;
3. **Gravar os trechos** → carga idempotente, pelo `id` do trecho;
4. **Buscar por similaridade** → a posição do trecho certo, e o HNSW (aproximado) contra o KNN exaustivo (exato);
5. **RAG completo** → a resposta com citação, a partir dos 3 primeiros trechos.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** um serviço **Azure AI Search** em qualquer tier. A busca vetorial roda até no Free (limitado a 3 índices e 50 MB), e este notebook não usa o semantic ranker. O serviço Basic dos notebooks de `tipos-de-rag` serve: o índice daqui é outro (`cap05-base-vetorial`), e o `cap05-acervo` não é tocado. Use a **chave de administração**, porque o notebook cria o índice. Não há opção a habilitar no serviço; só serviços criados antes de janeiro de 2019 podem não aceitar índice vetorial.

## 0. Preparação

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na Execução 5)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`
- `AZURE_SEARCH_ENDPOINT` e `AZURE_SEARCH_API_KEY` (chave de **administração**: este notebook cria um índice)

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "azure-search-documents>=12.0.0,<13"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
# Dois clientes: um administra o índice (criar, apagar), o outro grava e consulta documentos.
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT") or input(
    "AZURE_SEARCH_ENDPOINT (ex.: https://meu-servico.search.windows.net): "
).strip()
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY") or getpass("AZURE_SEARCH_API_KEY (chave de administração): ")

# Índice próprio desta demo: o cap05-acervo, dos notebooks de tipos-de-rag, não é tocado.
NOME_INDICE = "cap05-base-vetorial"
credencial_busca = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(AZURE_SEARCH_ENDPOINT, credencial_busca)
search_client = SearchClient(AZURE_SEARCH_ENDPOINT, NOME_INDICE, credencial_busca)

print("AI Search:", AZURE_SEARCH_ENDPOINT, "| índice:", NOME_INDICE)

### Funções auxiliares

As mesmas dos outros notebooks do Cap. 5 que as usam, sem mudar uma linha: nesta série, só a base muda.

- `carregar_trechos`, `texto_para_embedding` e `gerar_embeddings`: um trecho por seção `## `, o mesmo texto vetorizado, o mesmo modelo e a mesma dimensão de embedding.
- `PERGUNTAS` e `posicao`: o gabarito (`acervo/perguntas.json`) diz qual trecho responde cada pergunta; `posicao` diz em que lugar ele apareceu.
- `responder`: a geração com citação, usada na Execução 5.

In [ ]:
import json
import re

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


def texto_para_embedding(trecho: dict) -> str:
    """O texto que vira vetor: título do documento, seção e conteúdo."""
    return f"{trecho['titulo']} — {trecho['secao']}\n{trecho['conteudo']}"


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}


def posicao(ids_retornados: list[str], grupo: list[str]) -> int | None:
    """Posição (1, 2, 3...) do primeiro trecho esperado na lista retornada, ou None se ele não apareceu."""
    for i, id_trecho in enumerate(ids_retornados, start=1):
        if id_trecho in grupo:
            return i
    return None


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text

---

# Execução 1 — Preparar os vetores

Os trechos e os vetores são os mesmos nas seis bases desta série: um trecho por seção do acervo e um embedding por trecho, com o modelo e a dimensão do `.env`. Daqui para a frente, **só muda onde o vetor mora**.

In [ ]:
trechos = carregar_trechos(PASTA_ACERVO)
vetores = gerar_embeddings([texto_para_embedding(t) for t in trechos])

print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")
print(f"{len(vetores)} vetores, cada um com {len(vetores[0])} dimensões")

---

# Execução 2 — Criar a estrutura vetorial

No AI Search, a estrutura vetorial é um **índice**: campos de texto, um campo vetorial e a seção `vectorSearch`, que diz como os vetores são organizados.

- **Campos de texto** `titulo`, `secao` e `conteudo`, com o analisador `pt-BR.lucene`, e `fonte`, para citar a origem. O `id` do trecho é a chave.
- **Campo vetorial** `embedding`: `Collection(Edm.Single)` com `EMBEDDING_DIMENSIONS` dimensões, ligado ao perfil `perfil-vetorial`. Precisa ser `SearchField`: `SimpleField` e `SearchableField` ignoram a dimensão sem avisar.
- **Dois algoritmos**, as duas opções da linha do AI Search no slide 22:
  - `hnsw` é o algoritmo do perfil e, portanto, do campo: um grafo em camadas que encontra vizinhos **aproximados**. Os três parâmetros, escritos com os valores padrão, são os botões do slide 23: `m` (ligações por nó, de 4 a 10), `efConstruction` (candidatos na indexação, de 100 a 1.000) e `efSearch` (candidatos na consulta, de 100 a 1.000). Valores maiores tendem a subir o recall e cobram em memória, tempo de indexação e latência.
  - `knn-exaustivo` é força bruta: compara a pergunta com todos os vetores e devolve os vizinhos **exatos**. Fica declarado, mas **nenhum perfil aponta para ele**: para usá-lo, um campo precisaria de um perfil próprio com esse algoritmo e seria buscado sempre por força bruta. A comparação exata da Execução 4 não passa por ele: usa o parâmetro de consulta `exhaustive=True`, que roda o KNN exaustivo no próprio campo HNSW. O contrário não vale: um campo indexado com `exhaustiveKnn` não tem o grafo e não aceita consulta HNSW.
- **Sem vetorizador e sem configuração semântica**, de propósito: a pergunta vira vetor no cliente, com o mesmo `gerar_embeddings`. Busca híbrida e semantic ranker existem neste serviço, "sem código", e estão nos notebooks de `tipos-de-rag`.

### Os limites desta base (slide 22)

| Linha do AI Search no slide 22 | O que isso quer dizer aqui |
|---|---|
| Dimensões: até 4.096 | Limite por campo vetorial, em todos os tiers; o `text-embedding-3-small` vai até 1.536 |
| Índices: HNSW, KNN exaustivo | Os dois algoritmos acima, com a métrica `cosine`, a dos embeddings do Azure OpenAI |
| Reranking: sim | O semantic ranker: uma configuração semântica no índice e um parâmetro na consulta (`tipos-de-rag/03`) |

### O algoritmo do índice (slide 23)

O trade-off é recall, memória e tempo de indexação. O HNSW tem recall alto e lida bem com dado que muda, mas mantém **todos os vetores em memória**, e isso consome a cota de índice vetorial do serviço. O KNN exaustivo não consome essa cota e é exato, mas cada consulta percorre todos os vetores: serve para acervos pequenos e médios, ou para medir o recall do HNSW.

In [ ]:
from azure.search.documents.indexes.models import (
    ExhaustiveKnnAlgorithmConfiguration,
    ExhaustiveKnnParameters,
    HnswAlgorithmConfiguration,
    HnswParameters,
    LexicalAnalyzerName,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SimpleField,
    VectorSearch,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

campos = [
    SimpleField(name="id", type=SearchFieldDataType.STRING, key=True, filterable=True),
    SearchableField(name="titulo", analyzer_name=LexicalAnalyzerName.PT_BR_LUCENE),
    SearchableField(name="secao", analyzer_name=LexicalAnalyzerName.PT_BR_LUCENE),
    SearchableField(name="conteudo", analyzer_name=LexicalAnalyzerName.PT_BR_LUCENE),
    SimpleField(name="fonte", type=SearchFieldDataType.STRING, filterable=True),
    # O campo vetorial precisa ser SearchField: SimpleField e SearchableField ignoram a dimensão sem avisar.
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.SINGLE),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIMENSIONS,
        vector_search_profile_name="perfil-vetorial",
    ),
]

cosseno = VectorSearchAlgorithmMetric.COSINE
busca_vetorial = VectorSearch(
    algorithms=[
        # Os valores padrão do HNSW, escritos para mostrar os botões do triângulo do slide 23.
        HnswAlgorithmConfiguration(
            name="hnsw",
            parameters=HnswParameters(m=4, ef_construction=400, ef_search=500, metric=cosseno),
        ),
        ExhaustiveKnnAlgorithmConfiguration(name="knn-exaustivo", parameters=ExhaustiveKnnParameters(metric=cosseno)),
    ],
    # O perfil liga o campo a um algoritmo: aqui, o HNSW. Nenhum perfil usa o knn-exaustivo.
    profiles=[VectorSearchProfile(name="perfil-vetorial", algorithm_configuration_name="hnsw")],
)

indice = SearchIndex(name=NOME_INDICE, fields=campos, vector_search=busca_vetorial)
indice_criado = index_client.create_or_update_index(indice)

definicao = indice_criado.as_dict()
campo = next(c for c in definicao["fields"] if c["name"] == "embedding")
perfis = {p["name"]: p["algorithm"] for p in definicao["vectorSearch"]["profiles"]}
algoritmo_do_campo = perfis[campo["vectorSearchProfile"]]

print(f"Índice '{indice_criado.name}' criado ou atualizado. Como o serviço o registrou:")
print(f"Campo '{campo['name']}': {campo['type']}, {campo['dimensions']} dimensões, perfil '{campo['vectorSearchProfile']}'")
display(pd.DataFrame([
    {
        "algoritmo": a["name"],
        "tipo": a["kind"],
        "parâmetros": a.get("hnswParameters") or a.get("exhaustiveKnnParameters"),
        "usado pelo campo?": "sim" if a["name"] == algoritmo_do_campo else "não",
    }
    for a in definicao["vectorSearch"]["algorithms"]
]))

---

# Execução 3 — Gravar os trechos

Cada trecho entra no índice com o seu vetor, pela chave `id` (documento + número da seção). `merge_or_upload_documents` atualiza o documento quando a chave já existe e cria quando não existe: rodar o notebook de novo não duplica nada.

A contagem do índice leva alguns segundos para refletir a carga; por isso esperamos até ela chegar ao número de trechos.

In [ ]:
import time

documentos = [
    {"id": t["id"], "titulo": t["titulo"], "secao": t["secao"], "conteudo": t["conteudo"], "fonte": t["fonte"], "embedding": v}
    for t, v in zip(trechos, vetores)
]
resultado_carga = search_client.merge_or_upload_documents(documents=documentos)

falhas = [r.key for r in resultado_carga if not r.succeeded]
print(f"Enviados: {len(resultado_carga)} | com sucesso: {len(resultado_carga) - len(falhas)} | falhas: {len(falhas)}")
if falhas:
    print("Trechos com falha:", falhas)

# A contagem é atualizada com atraso: esperamos até ~15 segundos.
for _ in range(15):
    total_no_indice = search_client.get_document_count()
    if total_no_indice >= len(trechos):
        break
    time.sleep(1)

print(f"Trechos gravados no índice: {total_no_indice} (trechos no acervo: {len(trechos)})")
if total_no_indice > len(trechos):
    print("Há documentos a mais: sobras de uma carga anterior. merge_or_upload_documents não apaga nada.")

---

# Execução 4 — Buscar por similaridade

A pergunta vira vetor **no cliente**, com o mesmo `gerar_embeddings` da carga, e o índice devolve os `k` trechos mais próximos (`VectorizedQuery`). A **similaridade** é o próprio `@search.score`: com a métrica cosseno, o AI Search devolve `1 / (1 + distância de cosseno)`, que vai de 0,333 a 1. Quanto maior, mais próximo; não é probabilidade de acerto. A coluna **cosseno** desfaz essa transformação (`2 - 1 / similaridade`, a partir da fórmula do Microsoft Learn), e a **diferença para o 1º** é calculada sobre o cosseno: as duas colunas estão na escala das outras bases da série.

Duas perguntas do gabarito: uma em linguagem natural, em que o vetor tende a acertar, e a do **SKU 48213-B**, cujo trecho tem dois "irmãos" quase idênticos, o 48213-C e o 48213-D. Depois, a comparação: a pergunta do SKU vira vetor uma vez, e esse mesmo vetor vai ao HNSW (aproximado) e ao KNN exaustivo (`exhaustive=True`, exato), no mesmo campo.

### O que esperar e por que pode variar

- Na pergunta em linguagem natural, o trecho "Como pedir o reembolso de uma viagem" tende a vir no topo.
- No SKU, os três SKUs tendem a aparecer juntos, com similaridades muito próximas: o trecho certo pode ganhar por pouco, ou perder para um vizinho.
- O HNSW e o exaustivo tendem a devolver o mesmo top 5. Com só 50 trechos e `efSearch` = 500, o grafo tende a enxergar o acervo inteiro; a diferença tende a aparecer em acervos grandes.
- Na comparação, mesma pergunta, mesmo vetor: só o algoritmo muda. Se um trecho trocar de lugar entre o HNSW e o exaustivo, a causa é a aproximação, não o embedding.
- Os números dependem do modelo e da dimensão do embedding e podem mudar um pouco a cada vez que a pergunta vira vetor, inclusive entre a tabela do SKU e a comparação, que vetoriza a pergunta de novo.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def buscar(pergunta: str, k: int = 5, exaustivo: bool = False, vetor: list[float] | None = None) -> list[dict]:
    """Vetoriza a pergunta no cliente e pede ao índice os k vizinhos mais próximos.

    exaustivo=False usa o grafo HNSW (aproximado); exaustivo=True compara com todos os vetores (exato).
    vetor: o vetor da pergunta, quando ele já foi calculado (a comparação abaixo usa o mesmo nas duas buscas).
    """
    if vetor is None:
        vetor = gerar_embeddings([pergunta])[0]
    resultados = search_client.search(
        search_text=None,
        vector_queries=[VectorizedQuery(vector=vetor, k_nearest_neighbors=k, fields="embedding", exhaustive=exaustivo)],
        select=["id", "titulo", "secao", "conteudo", "fonte"],
        top=k,
    )
    # Com a métrica cosseno, o @search.score já é uma similaridade: 1 / (1 + distância de cosseno).
    return [
        {
            "id": r["id"],
            "titulo": r["titulo"],
            "secao": r["secao"],
            "conteudo": r["conteudo"],
            "fonte": r["fonte"],
            "similaridade": r["@search.score"],
        }
        for r in resultados
    ]


resultados_gabarito = {pid: buscar(PERGUNTAS[pid]["pergunta"], k=5) for pid in ["linguagem-natural", "string-exata"]}

for pid, top5 in resultados_gabarito.items():
    esperados = PERGUNTAS[pid]["trechos_esperados"][0]
    lugar = posicao([t["id"] for t in top5], esperados)
    situacao = f"{lugar}º lugar" if lugar else "fora do top 5"
    cossenos = [2 - 1 / t["similaridade"] for t in top5]  # desfaz o 1 / (1 + distância de cosseno)
    display(Markdown(f"**{PERGUNTAS[pid]['pergunta']}** — posição do trecho certo: {situacao}"))
    display(pd.DataFrame([
        {
            "posição": i,
            "similaridade": round(t["similaridade"], 4),
            "cosseno": round(c, 4),
            "diferença para o 1º": round(c - cossenos[0], 4),  # na escala do cosseno, como nas outras bases
            "trecho": f"{t['titulo']} — {t['secao']}",
            "é o trecho certo?": "sim" if t["id"] in esperados else "",
        }
        for i, (t, c) in enumerate(zip(top5, cossenos), start=1)
    ]))

In [ ]:
pergunta = PERGUNTAS["string-exata"]["pergunta"]
vetor = gerar_embeddings([pergunta])[0]  # um vetor só, para as duas buscas: só o algoritmo muda
aproximado = buscar(pergunta, k=5, vetor=vetor)  # HNSW: o algoritmo do perfil do campo
exato = buscar(pergunta, k=5, exaustivo=True, vetor=vetor)  # KNN exaustivo: compara com todos os vetores

display(Markdown(f"**{pergunta}** — HNSW (aproximado) x KNN exaustivo (exato), no mesmo campo e com o mesmo vetor"))
display(pd.DataFrame([
    {
        "posição": i,
        "HNSW": a["secao"],
        "similaridade HNSW": round(a["similaridade"], 4),
        "exaustivo": e["secao"],
        "similaridade exaustivo": round(e["similaridade"], 4),
        "mesmo trecho?": "sim" if a["id"] == e["id"] else "não",
    }
    for i, (a, e) in enumerate(zip(aproximado, exato), start=1)
]))

em_comum = len({t["id"] for t in aproximado} & {t["id"] for t in exato})
print(f"Recall@5 do HNSW, medido contra o exaustivo: {em_comum}/{len(exato)}")

### O que observar

- Nos SKUs, olhe a coluna **diferença para o 1º**: as descrições dos três "irmãos" são quase idênticas (quase só mudam o código, o modelo do rolamento, a vedação e os números), e o vetor tende a colocá-las lado a lado. É o mesmo quase empate do slide 14 e do notebook `tipos-de-rag/01`: **trocar a base não muda a natureza do vetor.** Lá, essa coluna está na escala do `@search.score` (`1 / (1 + distância)`), em que a mesma diferença parece de 2 a 3 vezes menor: compare a ordem, não os números.
- Para comparar com as outras bases da série, use as colunas **cosseno** e **diferença para o 1º**, que está na escala do cosseno, como nas outras bases. Com os mesmos vetores, o cosseno é o mesmo em qualquer base; muda só a escala em que cada uma devolve o resultado (distância, similaridade ou, aqui, `1 / (1 + distância)`).
- O recall@5 do HNSW contra o exaustivo tende a dar 5/5 aqui. Busca vetorial é **aproximada por definição** (slide 23): o exaustivo é o gabarito exato para medir quanto o HNSW perde, e é esse número que se acompanha quando o acervo cresce.

### Leitura arquitetural

- Ligar o KNN exaustivo em toda consulta resolveria a aproximação, não o quase empate: o exato também vê três SKUs quase iguais.
- O remédio para o SKU é de recuperação, não de base: o BM25 da busca híbrida acha a string exata que o vetor aproxima (slide 14). Busca híbrida e reranking são decisões de **recuperação** (`tipos-de-rag`); a base é decisão de **operação** (slide 21).

---

# Execução 5 — RAG completo

Os 3 primeiros trechos da pergunta em linguagem natural vão para o modelo, que responde citando o `id` de cada trecho usado. O `responder` é o mesmo das outras bases da série e de `tipos-de-rag`: a geração não sabe, nem precisa saber, em que base o vetor mora.

In [ ]:
pid = "linguagem-natural"
pergunta = PERGUNTAS[pid]["pergunta"]
top3 = resultados_gabarito[pid][:3]
lugar = posicao([t["id"] for t in top3], PERGUNTAS[pid]["trechos_esperados"][0])
enviados = ", ".join(f"`{t['id']}`" for t in top3)
situacao = f"sim, em {lugar}º lugar" if lugar else "não"

display(Markdown(f"### {pergunta}"))
display(Markdown(f"**Trechos enviados ao modelo:** {enviados}  \n**O trecho certo está entre eles?** {situacao}"))
display(Markdown(responder(pergunta, top3)))

### O que observar

- Leia a linha **O trecho certo está entre eles?** antes de julgar a resposta: ela separa erro de recuperação de erro de geração (slide 29).

### Leitura arquitetural

- Linha do AI Search no slide 22: **"Acervo grande e estável; híbrido e reranking sem código"**. É a resposta "Não, o acervo é estável" do slide 21: um índice dedicado ganha, porque foi construído para recuperar.
- "Sem código" quer dizer configuração. A busca híbrida (`tipos-de-rag/02`) é a mesma chamada `search_client.search()` com `search_text` **e** `vector_queries`: o serviço roda texto (BM25) e vetor e funde os resultados por RRF, e neste índice ela já funciona, porque os campos de texto são pesquisáveis. O semantic ranker (`tipos-de-rag/03`) pede uma configuração semântica no índice e `query_type="semantic"` na consulta.
- O preço é mais um componente para **operar, monitorar e pagar**. O índice é uma cópia do acervo (slide 24): alguém mantém a carga em dia (aqui, `merge_or_upload_documents`; ou um indexer agendado), monitora a cota de memória vetorial que o HNSW consome e paga as unidades de busca (réplicas × partições) por hora, enquanto o serviço existir.
- Se o dado muda em tempo real (pedido, estoque, cadastro), cada mudança precisa chegar ao índice. Aí o slide 21 põe o vetor junto do dado operacional, no banco que já existe: Cosmos DB, PostgreSQL e Azure SQL (pastas 02 a 04 desta série). Redis e DocumentDB (05 e 06) entram por outros critérios do slide 22: latência de milissegundos, ou um time que já vive em Mongo.

### Fala sugerida

> “A pergunta não é qual banco vetorial é o melhor. É se o dado muda em tempo real.”

---

# Quadro final — o que muda ao trocar a base?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | Os mesmos trechos e vetores de toda a série | O modelo e a dimensão do embedding são o contrato, qualquer que seja a base |
| 2 | Um índice com campo vetorial (até 4.096 dimensões), o HNSW no perfil e o KNN exaustivo declarado | O algoritmo: recall, memória e tempo de indexação (slide 23) |
| 3 | A carga roda de novo sem duplicar, pela chave `id` | Alguém precisa manter a cópia em dia com o acervo |
| 4 | O quase empate dos SKUs continua; HNSW e exaustivo tendem a empatar com 50 trechos | Trocar a base não conserta a recuperação; o exaustivo mede o recall do HNSW |
| 5 | A geração é a mesma, em qualquer base | A base é decisão de operação: o dado muda em tempo real? |

## Frase de fechamento

> **Escolher a base é decisão de operação, não de benchmark: cada componente novo precisa de alguém para operar, monitorar e pagar.**

---

# Referências oficiais

- Azure AI Search — criar um índice vetorial (campos, HNSW e KNN exaustivo):  
  https://learn.microsoft.com/azure/search/vector-search-how-to-create-index

- Azure AI Search — relevância e score na busca vetorial (`1 / (1 + distância de cosseno)`, KNN exaustivo como gabarito):  
  https://learn.microsoft.com/azure/search/vector-search-ranking

- Azure AI Search — criar uma consulta vetorial (`k` e `exhaustive`):  
  https://learn.microsoft.com/azure/search/vector-search-how-to-query

- Azure AI Search — limites por tier (4.096 dimensões por campo vetorial, cota de índice vetorial):  
  https://learn.microsoft.com/azure/search/search-limits-quotas-capacity

---

# Limpeza (opcional)

Nenhum outro notebook depende do índice `cap05-base-vetorial`. Com `APAGAR_RECURSOS = True`, a célula abaixo apaga **só esse índice**: o `cap05-acervo` e o serviço continuam. Apagar o índice não reduz a fatura de um tier dedicado: o serviço cobra por hora enquanto existir, e só apagar o serviço para a cobrança.

Mudou a dimensão do embedding no `.env`? Reinicie o kernel, porque os notebooks leem o `.env` só uma vez. Com a dimensão nova, a Execução 2 falha: a dimensão de um campo vetorial não muda depois de criada. Apague o índice aqui e rode o notebook inteiro de novo, com `APAGAR_RECURSOS` de volta em `False`.

Mudou só o modelo, com a mesma dimensão? Nada falha: o índice é o mesmo. Reinicie o kernel e rode o notebook inteiro, para regravar os 50 vetores com o modelo novo (slide 10: vetores de modelos diferentes não se comparam).

In [ ]:
APAGAR_RECURSOS = False  # True apaga só o índice desta demo

if APAGAR_RECURSOS:
    index_client.delete_index(NOME_INDICE)
    print(f"Índice '{NOME_INDICE}' apagado. O serviço e os outros índices continuam.")
else:
    print(f"Nada foi apagado. O índice '{NOME_INDICE}' continua no serviço.")